# Multi-Asset Trend & Beta Rotation Engine v4
## Long-only / Prop-oriented / Indices + Mega-cap Stocks + BTC/SOL

Pipeline: Universe → Liquidity → Momentum/Trend → Relative Strength → Beta Rotation → Correlation-aware Top N → Dynamic Sizing → Drawdown Governor → Rebalance → Live Tracker → Historical DB → Walk-forward research.

This is a research/paper-trading framework. Verify current FTMO rules separately before deployment.

In [ ]:
# Core configuration
import sqlite3, warnings
from datetime import datetime, timezone
import numpy as np, pandas as pd
import yfinance as yf

warnings.filterwarnings('ignore')
CONFIG = {'momentum_1m':21,'momentum_6m':126,'trend_fast':50,'trend_slow':200,'vol_window':30,'corr_window':90,'portfolio_size':8,'max_single_weight':0.20,'target_annual_vol':0.10,'max_portfolio_dd':0.10,'risk_on_scale':1.0,'neutral_scale':0.65,'risk_off_scale':0.35,'ratio_window':126,'risk_on_threshold':0.50,'risk_off_threshold':-0.50,'db_path':'multi_asset_trend_beta_rotation.db'}
INDEX_UNIVERSE=['SPY','QQQ','DIA','IWM']
MEGA_CAP_STOCKS=['AAPL','MSFT','AMZN','GOOGL','META','NFLX','TSLA','NVDA']
CRYPTO=['BTC-USD','SOL-USD']
UNIVERSE=INDEX_UNIVERSE+MEGA_CAP_STOCKS+CRYPTO
REGIME=['XLY','XLP','XLU','SPY','RSP']
UNIVERSE

## 1. Why combine indices, stocks and crypto?

Yes. They can compete in one ranking universe, but position sizing must account for their different volatility and correlations. The portfolio remains **long-only**: risk-off means reducing exposure and holding cash, not opening shorts.

The attached source recommends using beta rotations as a meta-layer/voting system rather than a dictator, starting with XLY/XLP, XLU/SPY and RSP/SPY. fileciteturn0file0L327-L345

In [ ]:
# Download daily adjusted prices
tickers=sorted(set(UNIVERSE+REGIME))
prices=yf.download(tickers,period='3y',interval='1d',auto_adjust=True,progress=False)['Close']
if isinstance(prices,pd.Series): prices=prices.to_frame()
prices=prices.ffill().dropna(axis=1,how='all')
returns=prices.pct_change()
prices.tail()

## 2. Core trend/momentum engine

We intentionally use few parameters: 1-month momentum, 6-month momentum, 50/200 trend, relative strength versus SPY and volatility. This follows the source's low-parameter philosophy. The source describes rotation as ranking assets by momentum and rotating into stronger, relatively uncorrelated assets. fileciteturn0file0L13-L21

In [ ]:
def pct(s,higher=True):
    r=s.rank(pct=True)
    return r if higher else 1-r

def build_factors(prices,returns):
    rows=[]
    spy_mom=prices['SPY'].pct_change(CONFIG['momentum_6m']).iloc[-1]
    for s in [x for x in UNIVERSE if x in prices.columns]:
        p=prices[s]; r=returns[s]
        m1=p.pct_change(CONFIG['momentum_1m']).iloc[-1]
        m6=p.pct_change(CONFIG['momentum_6m']).iloc[-1]
        trend=p.rolling(CONFIG['trend_fast']).mean().iloc[-1]/p.rolling(CONFIG['trend_slow']).mean().iloc[-1]-1
        vol=r.rolling(CONFIG['vol_window']).std().iloc[-1]*np.sqrt(252)
        rows.append({'symbol':s,'momentum_1m':m1,'momentum_6m':m6,'trend':trend,'relative_strength':m6-spy_mom,'volatility':vol})
    return pd.DataFrame(rows).set_index('symbol')

raw=build_factors(prices,returns)
raw

In [ ]:
def score_core(raw):
    x=raw.copy()
    x['momentum_score']=0.35*pct(x.momentum_1m)+0.65*pct(x.momentum_6m)
    x['trend_score']=pct(x.trend)
    x['rs_score']=pct(x.relative_strength)
    x['vol_score']=pct(x.volatility,False)
    x['core_score']=0.40*x.momentum_score+0.25*x.trend_score+0.20*x.rs_score+0.15*x.vol_score
    x['long_eligible']=(x.momentum_6m>0)&(x.trend>0)
    x['rank']=x.core_score.rank(ascending=False,method='first').astype(int)
    return x.sort_values('rank')
scored=score_core(raw)
scored

## 3. Beta Rotation Meta-Layer

The source identifies XLY/XLP (consumer confidence/risk appetite), XLU/SPY (defensive flow) and RSP/SPY (breadth/concentration) as the practical starting set. fileciteturn0file0L65-L91 fileciteturn0file0L289-L307

We use ratio trend + rolling z-score. XLY/XLP and RSP/SPY rising are risk-on; XLU/SPY rising is risk-off. The three votes are averaged.

In [ ]:
def zscore(s,w):
    return (s-s.rolling(w).mean())/s.rolling(w).std().replace(0,np.nan)

ratios=pd.DataFrame({'XLY_XLP':prices['XLY']/prices['XLP'],'XLU_SPY':prices['XLU']/prices['SPY'],'RSP_SPY':prices['RSP']/prices['SPY']})
signals=pd.DataFrame(index=ratios.index)
signals['XLY_XLP']=zscore((ratios.XLY_XLP.rolling(21).mean()/ratios.XLY_XLP.rolling(CONFIG['ratio_window']).mean()-1),CONFIG['ratio_window'])
signals['XLU_SPY']=-zscore((ratios.XLU_SPY.rolling(21).mean()/ratios.XLU_SPY.rolling(CONFIG['ratio_window']).mean()-1),CONFIG['ratio_window'])
signals['RSP_SPY']=zscore((ratios.RSP_SPY.rolling(21).mean()/ratios.RSP_SPY.rolling(CONFIG['ratio_window']).mean()-1),CONFIG['ratio_window'])
signals['regime_score']=signals.mean(axis=1)
signals['regime_state']=np.select([signals.regime_score>=CONFIG['risk_on_threshold'],signals.regime_score<=CONFIG['risk_off_threshold']],['RISK_ON','RISK_OFF'],default='NEUTRAL')
latest_regime=signals.dropna().iloc[-1]
latest_regime

## 4. Correlation-aware Top N + dynamic sizing

Select long-eligible leaders, but penalize candidates that are highly correlated with already selected assets. Then size approximately by score/volatility and cap any single position at 20%.

In [ ]:
def corr_penalty(symbol,selected,returns):
    if not selected:return 1.0
    peers=[s for s in selected if s in returns.columns]
    if not peers:return 1.0
    c=returns[[symbol]+peers].tail(CONFIG['corr_window']).corr()[symbol].drop(symbol).abs().mean()
    return float(np.clip(1-c if pd.notna(c) else 1.0,0.25,1.0))

def construct_portfolio(scored,returns,regime_state):
    c=scored[scored.long_eligible].copy()
    selected=[]
    while len(selected)<CONFIG['portfolio_size'] and len(selected)<len(c):
        available=[s for s in c.index if s not in selected]
        if not available:break
        best=max(available,key=lambda s:c.loc[s,'core_score']*corr_penalty(s,selected,returns))
        selected.append(best)
    p=c.loc[selected].copy()
    raw=p.core_score/p.volatility.clip(lower=0.10)
    w=(raw/raw.sum()).clip(upper=CONFIG['max_single_weight']); w=w/w.sum()
    scale={'RISK_ON':CONFIG['risk_on_scale'],'NEUTRAL':CONFIG['neutral_scale'],'RISK_OFF':CONFIG['risk_off_scale']}[regime_state]
    p['weight']=w*scale
    p['signal']='LONG'
    p['cash_weight']=1-p.weight.sum()
    return p

portfolio=construct_portfolio(scored,returns,latest_regime['regime_state'])
portfolio[['rank','core_score','momentum_6m','trend','volatility','weight','cash_weight']]

## 5. Drawdown governor — 10% is a hard research ceiling

The objective is **not** to run at 10% drawdown. Exposure should be reduced progressively before reaching that level. In this implementation: 2.5% DD → 75% scale; 5% → 50%; 7.5% → 25%; 10% → flat/cash.

For a prop account, the actual firm limits must be checked separately; this is a portfolio-risk governor, not a guarantee of compliance.

In [ ]:
def dd_governor(portfolio,returns):
    w=portfolio.weight.copy(); cols=[s for s in w.index if s in returns.columns]
    pr=(returns[cols].fillna(0)*w.loc[cols]).sum(axis=1)
    eq=(1+pr).cumprod(); dd=eq/eq.cummax()-1; current=float(dd.iloc[-1])
    a=abs(min(current,0))
    scale=0.0 if a>=0.10 else 0.25 if a>=0.075 else 0.50 if a>=0.05 else 0.75 if a>=0.025 else 1.0
    out=portfolio.copy(); out['final_weight']=out.weight*scale; out['final_weight']=out.final_weight
    return out,{'current_dd':current,'dd_scale':scale}

portfolio,dd_status=dd_governor(portfolio,returns)
portfolio[['weight','final_weight']],dd_status

## 6. Rebalance plan

Risk-off does not mean short. It means the target portfolio moves toward cash.

In [ ]:
def rebalance_plan(target,current=None,threshold=0.03):
    current=current or {}; rows=[]
    for s,r in target.iterrows():
        old=float(current.get(s,0)); new=float(r.final_weight); d=new-old
        action='ADD' if old==0 and new>0 else 'EXIT' if new==0 and old>0 else 'REBALANCE' if abs(d)>=threshold else 'HOLD'
        rows.append([s,old,new,d,action])
    for s,old in current.items():
        if s not in target.index:rows.append([s,old,0,-old,'EXIT'])
    return pd.DataFrame(rows,columns=['symbol','current_weight','target_weight','delta','action'])
rebalance_plan(portfolio)

## 7. Historical signal database

Store every scan so we can later test forward returns, rank acceleration, beta-regime effects, Top-5 vs Top-10, rebalance frequency and walk-forward robustness.

In [ ]:
conn=sqlite3.connect(CONFIG['db_path'])
conn.execute('CREATE TABLE IF NOT EXISTS asset_signals (timestamp TEXT,symbol TEXT,rank INTEGER,core_score REAL,momentum_1m REAL,momentum_6m REAL,trend REAL,relative_strength REAL,volatility REAL,regime_state TEXT,regime_score REAL,target_weight REAL)')
conn.execute('CREATE TABLE IF NOT EXISTS portfolio_snapshots (timestamp TEXT,symbol TEXT,weight REAL,cash_weight REAL,current_dd REAL,dd_scale REAL)')
conn.commit()

def save_snapshot(scored,portfolio,regime,dd):
    ts=datetime.now(timezone.utc).isoformat()
    for s,r in scored.iterrows():
        conn.execute('INSERT INTO asset_signals VALUES (?,?,?,?,?,?,?,?,?,?,?,?)',(ts,s,int(r['rank']),float(r['core_score']),float(r['momentum_1m']),float(r['momentum_6m']),float(r['trend']),float(r['relative_strength']),float(r['volatility']),regime['regime_state'],float(regime['regime_score']),float(portfolio.loc[s,'final_weight']) if s in portfolio.index else 0.0))
    for s,r in portfolio.iterrows():
        conn.execute('INSERT INTO portfolio_snapshots VALUES (?,?,?,?,?,?)',(ts,s,float(r['final_weight']),float(r['cash_weight']),float(dd['current_dd']),float(dd['dd_scale'])))
    conn.commit()
save_snapshot(scored,portfolio,latest_regime,dd_status)
print('snapshot saved')

## 8. Live tracker

Use this as a monitoring layer. Keep actual order execution separate and add hard controls for daily loss, total exposure, stale data, order reconciliation and kill-switch behavior.

In [ ]:
def live_tracker(symbols):
    rows=[]
    for s in symbols:
        try:
            q=yf.Ticker(s).fast_info; last=q.get('last_price',np.nan); prev=q.get('previous_close',np.nan)
            rows.append({'symbol':s,'last':last,'change_pct':(last/prev-1)*100 if prev else np.nan})
        except Exception: rows.append({'symbol':s,'last':np.nan,'change_pct':np.nan})
    return pd.DataFrame(rows)
live_tracker(UNIVERSE)

## 9. Research checklist

Compare: (1) no beta overlay vs exposure scaler, (2) combined universe vs stocks-only vs indices-only vs crypto-only, (3) equal-weight vs inverse-vol, (4) no DD governor vs progressive governor, (5) daily vs weekly vs biweekly rebalance.

Primary metrics for prop-oriented research: maximum DD, worst day, rolling Sharpe, Calmar, turnover, exposure, tail loss, recovery time. Do not select parameters solely by CAGR. Use a clean out-of-sample period and walk-forward testing.

The source also cautions against implementing every beta dimension at once and recommends starting with the three ratios used here. fileciteturn0file0L327-L345

## 10. Final architecture

`INDICES + STOCKS + BTC/SOL → MOMENTUM/TREND → RELATIVE STRENGTH → BETA ROTATION → CORRELATION-AWARE TOP N → INVERSE-VOL SIZING → DD GOVERNOR → CASH WHEN RISK-OFF → REBALANCE → DATABASE → WALK-FORWARD VALIDATION`

**Principle:** trend engine selects the leaders; beta rotation controls aggressiveness; correlation controls concentration; drawdown governor controls total risk.